[`loop.py`](https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/loops.py)

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [12]:
from __future__ import annotations

import os
import time
import copy
from functools import partial
from typing import Optional, Union

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch import distributed as torch_dist

import numpy as np

from mmengine import Config, DictAction

import sys

sys.path.append('../../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.slowfast.mmaction.utils import SampleList
from computer_vision.slowfast.mmaction.models.recognizers.recognizer3d import Recognizer3D

from computer_vision.slowfast.mmaction.datasets.transforms.loading import DecordInit, SampleFrames, DecordDecode
from computer_vision.slowfast.mmaction.datasets.transforms.processing import Resize, _init_lazy_of_proper, RandomCrop, CenterCrop, ThreeCrop, \
RandomResizedCrop, Flip
from computer_vision.slowfast.mmaction.datasets.transforms.formatting import FormatShape, PackActionInputs
from computer_vision.slowfast.mmengine.dataset.base_dataset import Compose
from computer_vision.slowfast.mmengine.dataset.utils import pseudo_collate, worker_init_fn

from computer_vision.slowfast.parameter_parser import parser, merge_args
from computer_vision.slowfast.mmaction.datasets.video_dataset import VideoDataset
from computer_vision.slowfast.mmengine.dataset.sampler import DefaultSampler
from computer_vision.slowfast.mmengine.evaluator.evaluator import Evaluator

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
config="../../config/slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb.py"

data_dirpath='D:/data/UCF101'
seed=1
# root=f'{data_dirpath}/UCF-101'
# annotation_path=f'{data_dirpath}/UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
annotation_path='UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
metadata_path=f'{data_dirpath}/metadata.pt'

mini_train=False
if not mini_train:
    output_dirpath="D:/results/ucf101/mmaction2-slowfast/train"
    arguments= f"""-d {data_dirpath} -a {annotation_path} 
    """ # --use-cutmix-mixup
else:
    output_dirpath="D:/results/ucf101/mmaction2-slowfast/mini_train"
    arguments= f"""-d {data_dirpath} -a {annotation_path}
    """ # --use-cutmix-mixup --time 18
    
arguments+=f"""--data-prefix UCF-101 {config} --work-dir {output_dirpath} --auto-scale-lr --seed 1"""


args=parser.parse_args(arguments.split())
cfg=Config.fromfile(args.config)
args=merge_args(cfg, args)



In [11]:
pipeline=[DecordInit(io_backend='disk'),
            SampleFrames(clip_len=32, frame_interval=2, num_clips=1),
            DecordDecode(),
            Resize(scale=(-1, 256), keep_ratio=True, interpolation='bilinear', lazy=False),
            RandomResizedCrop(),
            Resize(scale=(224, 224), keep_ratio=False, interpolation='bilinear', lazy=False),
            Flip(flip_ratio=.5),
            FormatShape(input_format='NCTHW'),
            PackActionInputs()]
# we note here multi_class is just to inform the class to compute onehot
dataset=VideoDataset(ann_file=args.ann_file, pipeline=pipeline, data_root=args.data_root, data_prefix=dict(video=args.data_prefix), multi_class=False, 
                 num_classes=None, start_index=0, modality='RGB', test_mode=False, delimiter=' ', lazy_init=False)
sampler=DefaultSampler(dataset, shuffle=True, seed=seed, round_up=True)
init_fn=partial(worker_init_fn, num_workers=8, rank=0, seed=seed)
dataloader=DataLoader(dataset=dataset, batch_size=8, sampler=sampler, num_workers=8, persistent_workers=True,
                  collate_fn=pseudo_collate, worker_init_fn=init_fn)

runner=None
training_loop=EpochBasedTrainingLoop(runner, dataloader=dataloader, max_epochs=256, val_begin=1, val_interval=5, dynamic_intervals=None)

In [13]:


val_loop=ValLoop(runner, dataloader=dataloader, evaluator=Evaluator(), fp16=False)
        